In [2]:
import sleap
sleap.disable_preallocation()
sleap.versions()
sleap.system_summary()
import cv2
import sys, os
import numpy as np
from copy import copy
sys.path.append('../..')

SLEAP: 1.4.1
TensorFlow: 2.7.0
Numpy: 1.21.5
Python: 3.7.12
OS: Linux-6.8.0-51-generic-x86_64-with-debian-trixie-sid
GPUs: 2/2 available
  Device: /physical_device:GPU:0
         Available: True
       Initialized: False
     Memory growth: True
  Device: /physical_device:GPU:1
         Available: True
       Initialized: False
     Memory growth: True


In [5]:
from load_data_10min import load_video
from python.postprocess import *
from python.animation import *

In [3]:
original_video_path = '/home/mingxiao/Desktop/jellyfish/video/video_1_clips/c1_high_res_10min_track_reencoded.mp4'
# new_video_path = '/home/mingxiao/Desktop/jellyfish/video/video_1_clips/c1_high_res_5min_track_reencoded.mp4'
original_video = load_video(original_video_path, load_as_tensor=False)
# new_video = load_video(new_video_path, load_as_tensor=False)

print(original_video.shape)
# print(new_video.shape)

(90003, 170, 174)


In [38]:
def find_start_end_idx(original_video, new_video):
    new_vid_first_frame = new_video[0]
    new_vid_last_frame = new_video[-1]
    for i in range(original_video.shape[0]):
        if np.array_equal(original_video[i], new_vid_first_frame):
            start_idx = i
        elif np.array_equal(original_video[i], new_vid_last_frame):
            end_idx = i
            break
    print(start_idx, end_idx)
    return start_idx, end_idx

In [10]:
new_vid_first_frame = new_video[0]
new_vid_last_frame = new_video[-1]
for i in range(original_video.shape[0]):
    if np.array_equal(original_video[i], new_vid_first_frame):
        start_idx = i
    elif np.array_equal(original_video[i], new_vid_last_frame):
        end_idx = i
        break
print(start_idx, end_idx)

2850 47849


In [5]:
# start_idx, end_idx = 2977, 47935

In [11]:
old_points_path = '/home/mingxiao/Desktop/jellyfish/video/video_1_clips/all_tracked_points_raw_0.npy'
old_points = np.load(old_points_path)
new_points = old_points[start_idx:end_idx+1]
print(new_points.shape)

(45000, 17, 2)


In [3]:
reencoded_10min_path = '/home/mingxiao/Desktop/jellyfish/video/video_1_clips/predictions/flowmax-tracking/reencoded_10min_t3.slp'
reencoded_10min_labels = sleap.load_file(reencoded_10min_path)
print(reencoded_10min_labels)

Labels(labeled_frames=90003, videos=1, skeletons=1, tracks=18)


In [33]:
new_tracks = [track for track in reencoded_10min_labels.tracks if track.name != 'track_17']
new_tracks[0]

Track(spawned_on=0, name='track_0')

In [21]:
v2_5min_path = '/home/mingxiao/Desktop/jellyfish/video/video_1_clips/reencoded_5min_v2.slp'
v2_5min = sleap.load_file(v2_5min_path)

In [5]:
all_points = get_all_tracked_points(v2_5min, interpolate=False, min_score=0, start_idx=0)

all_tracked_points shape: (45000, 17, 2)
First non missing frame idx: 0
Missing point count: 72608
reorder_idx.shape: (17,)


In [32]:
new_model_with_new_points = dataset_with_new_points(new_model, new_points, node_name='tb', handle_first_frame=False)
print(new_model_with_new_points)

Skeleton(description=None, nodes=[tb], edges=[], symmetries=[])
17
missing_pt_cnt: 72752
Labels(labeled_frames=45000, videos=1, skeletons=1, tracks=17)


In [37]:
save_path = '/home/mingxiao/Desktop/jellyfish/video/video_1_clips/reencoded_5min_v2.slp'
new_model_with_new_points.save(save_path)

In [ ]:
new_tracks = [track for track in reencoded_10min_labels.tracks if track.name != 'track_17']

for i in range(5):
        
    new_dataset_path = f'/home/mingxiao/Desktop/jellyfish/video/video_1_clips/reencoded_5min_c{i}.slp'
    new_dataset = sleap.load_file(new_dataset_path)
    # skeleton = sleap.Skeleton(name=f'TB')
    # skeleton.add_node(f'tb')
    # new_dataset.skeletons = [skeleton]
    new_dataset.tracks = new_tracks
    
    new_points = all_points[i * 9000 : (i + 1) * 9000]
    
    new_dataset_w_pts = dataset_with_new_points(new_dataset, new_points, node_name='tb1_node', handle_first_frame=False, start_idx=0)
    new_dataset_w_pts.save(new_dataset_path)


## create dataset from code

In [7]:
c0_video_path = '/home/mingxiao/Desktop/jellyfish/video/video_1_clips/c1_high_res_5min_track_reencoded_0.mp4'
c0_video = sleap.Video.from_media(c0_video_path)
print(c0_video)

Video(filename=/home/mingxiao/Desktop/jellyfish/video/video_1_clips/c1_high_res_5min_track_reencoded_0.mp4, shape=(9000, 170, 174, 1), backend=MediaVideo)


In [9]:
print(v2_5min.videos[0])

Video(filename=/home/mingxiao/Desktop/jellyfish/video/video_1_clips/c1_high_res_5min_track_reencoded.mp4, shape=(45000, 170, 174, 1), backend=MediaVideo)


In [ ]:
new_tracks = [track for track in reencoded_10min_labels.tracks if track.name != 'track_17']
new_dataset.tracks = new_tracks
skeleton = sleap.Skeleton(name=f'TB')
skeleton.add_node(f'tb')
new_dataset.skeletons = [skeleton]


In [12]:
def create_dataset(video_path, points):
    video = sleap.Video.from_media(video_path)
    skeleton = sleap.Skeleton(name=f'TB')
    skeleton.add_node(f'tb')
    frame_cnt = video.shape[0]
    inst_cnt = points.shape[1]
    track_lst = [None for _ in range(inst_cnt)]
    labeled_frames = []
    
    for frame_idx in range(frame_cnt):
        curr_frame = sleap.instance.LabeledFrame(video=video, frame_idx=frame_idx)
        inst_lst = []
        for inst_idx in range(inst_cnt):
            x, y = points[frame_idx, inst_idx]
            if x + y == 0:
                continue
            if track_lst[inst_idx] is None:
                track_lst[inst_idx] = sleap.Track(name=f'track_{inst_idx}', spawned_on=frame_idx)
                print(track_lst[inst_idx])
            point_dict = {'tb': sleap.instance.Point(x=x, y=y)}
            tb_instance = sleap.Instance(
                skeleton=skeleton, 
                points=point_dict, 
                frame=curr_frame, 
                track=track_lst[inst_idx])
            inst_lst.append(tb_instance)
        curr_frame.instances = inst_lst
        labeled_frames.append(curr_frame)
    
    dataset = sleap.io.dataset.Labels(
            labeled_frames=labeled_frames, 
            videos=[video],
            tracks=track_lst, 
            skeletons=[skeleton],
            nodes=[skeleton.nodes[0]])
    return dataset

In [22]:
for i in range(5):
    # video_path = f'/home/mingxiao/Desktop/jellyfish/video/video_1_clips/c1_high_res_5min_track_reencoded_{i}.mp4'
    # new_points = all_points[i * 9000 : (i + 1) * 9000]
    dataset = v2_5min.extract(slice(i * 9000, (i + 1) * 9000))
    dataset.save(f'/home/mingxiao/Desktop/jellyfish/video/video_1_clips/manual_5min_c{i}.slp')


In [2]:
for i in range(5):
    # vid_only_dataset_path = f'/home/mingxiao/Desktop/jellyfish/video/video_1_clips/reencoded_5min_c{i}.slp'
    # vid_only_dataset = sleap.load_file(vid_only_dataset_path)
    # vid = vid_only_dataset.videos[0]
    vid_path = f'/home/mingxiao/Desktop/jellyfish/video/video_1_clips/c1_high_res_5min_track_reencoded_{i}.mp4'
    vid = sleap.Video.from_media(vid_path)
    labeled_dataset_path = f'/home/mingxiao/Desktop/jellyfish/video/video_1_clips/manual_5min_c{i}.slp'
    labeled_dataset = sleap.load_file(labeled_dataset_path)
    
    for lf in labeled_dataset.labeled_frames:
        lf.video = vid
    labeled_dataset.videos = [vid]
    labeled_dataset.save(labeled_dataset_path)


In [15]:
test_dataset = sleap.load_file('/home/mingxiao/Desktop/jellyfish/video/video_1_clips/manual_5min_c0.slp')
test_dataset.video

Video(backend=MediaVideo(filename='/home/mingxiao/Desktop/jellyfish/video/video_1_clips/c1_high_res_5min_track_reencoded_0.mp4', grayscale=True, bgr=True, dataset='', input_format=''))

In [16]:
v2_5min.videos[0]

Video(backend=MediaVideo(filename='/home/mingxiao/Desktop/jellyfish/video/video_1_clips/c1_high_res_5min_track_reencoded.mp4', grayscale=True, bgr=True, dataset='', input_format=''))

## preprocess manually labeled dataset

In [6]:
manual_dataset_path = '/home/mingxiao/Desktop/jellyfish/video/video_1_clips/manual_5min_c0_copy.slp'
manual_dataset = sleap.load_file(manual_dataset_path)
manual_dataset

Labels(labeled_frames=9000, videos=1, skeletons=1, tracks=17)

In [7]:
manual_points = get_all_tracked_points(manual_dataset, interpolate=False, min_score=0, start_idx=0, reorder=False)
manual_points.shape

all_tracked_points shape: (9000, 17, 2)
First non missing frame idx: 0
Missing point count: 7492


(9000, 17, 2)

In [8]:
np.save('/home/mingxiao/Desktop/jellyfish/video/video_1_clips/manual_5min_c0_points.npy', manual_points)

In [6]:
34 * 9000 - 7492

298508